In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import darkprop as dp
from darkprop import units
from darkprop import constants

# %matplotlib notbook
%matplotlib inline

def events_to_arrays(events):
    t = []
    T = []
    r = []
    p = []
    for event in events:
        t.append(event.t)
        T.append(event.T)
        r.append(event.r.to_vec())
        p.append(event.p3.to_vec())
    return np.array(t), np.array(T), np.array(r), np.array(p)

def plot_trajectories(all_events):
    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')
    for events in all_events:
        t, T, r, p = events_to_arrays(events)
        x = r[:, 0]
        y = r[:, 1]
        z = r[:, 2] - 6371
        ax.scatter(x, y, z)
        ax.plot(x, y, z, ls='-')
    ax.set_xlabel('x [km]')
    ax.set_ylabel('y [km]')
    ax.set_zlabel('z [km]')
    return fig, ax

# Initialization

In [ ]:
# ==========================
# parameters
# ==========================

# DM mass
m = 0.01 * units.GeV  # note GeV = 1

# cross section
sigma0 = 1e-30 * units.cm**2


# initial kinetic energy and 3 momentum
T0 = 0.1 * units.GeV
p0 = np.sqrt(T0 * (T0 + 2 * m))
init_p = [0.0, 0.0, -p0]

# or initial velocity
vi = 220 * units.km / units.sec
init_v = [0.0, 0.0, -vi]


# initial position (x, y, z)
init_r = [0.0, 0.0, dp.constants.rEarth]


# ==========================
# Earth model initialization
# ==========================

# HomoEarth: homogeneous Earth composed of 8 components
earth = dp.HomoEarth()


# ==========================
# DM Particle initialization
# ==========================

# SIDM: spin-independent elasic scattering cross section without nuclear form factor
dm = dp.SIDM()

# SIHelmDM: spin-independent elasic scattering cross section with Helm form factor
# dm = dp.SIHelmDM()

# note that SIHelmDM needs this initialization
# for target in earth.getTargets():
#     dm.initCrossSectionCDF(1e-100, p0, target, 100000)

# set cross section and mass
dm.sigma0 = sigma0
dm.m = m

# set initial condition
dm.setR(init_r)
dm.setP(init_p)


# ==========================
# random number generator
# ==========================
rn = dp.RandomNumber()

# Simulate 10 trajectories with the same initial conditions

## DM-nucleus scattering

In [ ]:
%%time

# cutoff energy
# give a value
Tcut = 1e-5 * units.GeV

# or from a velocity
# vcut = 1.0 * cm / sec
# Tcut = 0.5 * m * vcut**2

# list of Event list
all_events = []
for i in range(10):
    # reset the initial conditions
    dm.setR(init_r)
    dm.setP(init_p)
    # dm.setV(ini_v)
    # reset time
    dm.t = 0
    dm.in_medium = True

    # use predefined simulate_track function
    # which return a list of Events
    all_events.append(dp.simulate_track(dm, earth, Tcut, rn))

fig, ax = plot_trajectories(all_events)
ax.set_title("DM-nucleus isotropic scattering")

## DM-electron scattering

In [ ]:
%%time

Tcut = 1e-5 * units.GeV

# HomoElectronEarth: homogeneous Earth with electron only
earth_electron = dp.HomoElectronEarth()


all_events = []
for i in range(10):
    # reset the initial conditions
    dm.setR(init_r)
    dm.setP(init_p)
    dm.t = 0
    dm.in_medium = True

    all_events.append(dp.simulate_track(dm, earth_electron, Tcut, rn))

fig, ax = plot_trajectories(all_events)
ax.set_title("DM-electron")

# Lower level interface

## DM-nuclues scattering with Helm form factor

In [ ]:
%%time
dm_ff = dp.SIHelmDM(sigma0, m)

T0 = 1 * units.GeV
p_init = [0, 0, -np.sqrt(T0**2 + 2 * m * T0)]

# differential cross section CDF interpolation
for target in earth.getTargets():
    dm_ff.initCrossSectionCDF(1e-50, T0, target, 100000)

In [ ]:
%%time
# 10 trajectories again but now include the form factor

Tcut = 1e-5 * units.GeV

all_events = []
for i in range(10):
    dm_ff.setR(init_r)
    dm_ff.setP(p_init)
    dm_ff.t = 0
    dm_ff.in_medium = True

    events = []
    # use propagate and scatter instead of simulate_track function
    while dm_ff.in_medium:
        dp.propagate(dm_ff, earth, rn)
        dp.scatter(dm_ff, earth, rn)
        events.append(dm_ff.toEvent())  # convert to Event object (default in unit of km, sec, GeV)
        if dm_ff.T < Tcut:
            dm_ff.in_medium = False

    all_events.append(events)

fig, ax = plot_trajectories(all_events)
ax.set_title("DM-nucleus with Helm form factor")

# Halo DM

In [ ]:
%%time
v_earth = [0, 0, 230*units.km/units.sec]
positions = []
velocities = []
speeds = []
for i in range(10000):
    dp.init_halo(
        dm,                     # DM particle
        0,                      # initial time
        1e-10,                  # vmin (c=1)
        v_earth,                # Earth's velocity
        rn,                     # random number generator
        constants.rEarth*1.1,   # distance of the initial disk from the Earth's center
    )  # for other default parameters please see the documentation

    dp.inject(dm)  # put it on the Earth's surface

    # get positon and velocity
    positions.append(np.array(dm.r.to_vec()) / units.km)
    velocities.append(np.array(dm.v.to_vec()) / (units.m / units.sec))
    speeds.append(np.linalg.norm(dm.v.to_vec()))

    # simulation can be done here...

positions = np.array(positions)
velocities = np.array(velocities)
speeds = np.array(speeds)

## Show initial positions

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
x = positions[:, 0]
y = positions[:, 1]
z = positions[:, 2]

ax.scatter(x, y, z, s=0.05)
ax.set_xlabel('x [km]')
ax.set_ylabel('y [km]')
ax.set_zlabel('z [km]')

## Plot normalized speed distribution

In [ ]:
speed_of_light = 2.99792458e5  # km / s
fig, ax = plt.subplots()
ax.hist(speeds * speed_of_light, bins=50, density=True)
ax.set_xlabel(r'$v$ [km]')
ax.set_ylabel(r'$f(v)$ [km$^{-1}$]')